# World-Intel-MCP — Google Colab Test Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marc-shade/world-intel-mcp/blob/master/notebooks/world_intel_mcp_test.ipynb)

A **self-contained** test notebook for the [World-Intel-MCP](https://github.com/marc-shade/world-intel-mcp) server.  
Tests news & RSS feed tools end-to-end and builds an Anthropic-powered intelligence agent.

---

## Table of Contents

| Section | Description |
|---------|-------------|
| **0. Installation** | Install `world-intel-mcp` + `anthropic` from GitHub |
| **1. Verification** | Confirm all dependencies are present and import correctly |
| **2. Infrastructure** | Unit-test Cache, CircuitBreaker, Fetcher components |
| **3. MCP Protocol** | Launch server subprocess, list 87+ tools, verify identity |
| **4. `intel_news_feed`** | Test RSS aggregation across categories with assertions |
| **5. `intel_trending_keywords`** | Keyword spike detection + live bar chart |
| **6. `intel_gdelt_search`** | GDELT 2.0 article list + volume timeline + chart |
| **7. NLP Tools** | Entity extraction, event classification, news clustering |
| **8. WorldIntelAgent** | Anthropic tool-use agent class backed by MCP |
| **9. Agent Demo** | Three live intelligence queries with agentic tool chaining |
| **Final Summary** | Pass/fail tally across all test assertions |

---

> **API keys required:**  
> Sections 0–7: **none** — all news, RSS, and GDELT tools are free & unauthenticated.  
> Sections 8–9: **`ANTHROPIC_API_KEY`** — add to Colab Secrets (🔑 icon) before running.


## Section 0 — Installation

Installs `world-intel-mcp` directly from GitHub plus the `anthropic` SDK for the agent demo.  
If this is a fresh Colab session, **Runtime → Restart session** after the cell completes, then run all cells again.


In [ ]:
# ─── Install world-intel-mcp from GitHub ─────────────────────────────────────
print('Installing world-intel-mcp from GitHub...')
%pip install -q "world-intel-mcp @ git+https://github.com/marc-shade/world-intel-mcp.git"

print('Installing anthropic + nest_asyncio...')
%pip install -q anthropic nest_asyncio

print()
print('✅ Installation complete.')
print('   If this is your first install in this session, Colab may need to restart.')
print('   Runtime → Restart session → then Run all cells.')


## Section 1 — Dependency Verification & Setup

Confirm all required packages are installed, then set up the async helper and test tracker used throughout the notebook.


In [ ]:
# ─── Dependency version table ─────────────────────────────────────────────────
import importlib.metadata
import sys

REQUIRED = {
    'world-intel-mcp': 'world-intel-mcp',
    'mcp':             'mcp',
    'httpx':           'httpx',
    'feedparser':      'feedparser',
    'pydantic':        'pydantic',
    'rich':            'rich',
    'click':           'click',
    'jinja2':          'Jinja2',
    'anthropic':       'anthropic',
    'nest_asyncio':    'nest-asyncio',
}

print(f'Python {sys.version.split()[0]}\n')
print(f'  {"Package":<22} {"Version":<14} Status')
print(f'  {"─"*22} {"─"*14} {"─"*6}')

all_ok = True
for label, pkg in REQUIRED.items():
    try:
        ver = importlib.metadata.version(pkg)
        print(f'  {label:<22} {ver:<14} ✅')
    except importlib.metadata.PackageNotFoundError:
        print(f'  {label:<22} {"N/A":<14} ❌ MISSING')
        all_ok = False

print()
if all_ok:
    print('✅ All dependencies present.')
else:
    print('❌ Missing packages — re-run Section 0.')


In [ ]:
# ─── Async helpers + test tracker ────────────────────────────────────────────
import asyncio
import json
import os
from datetime import datetime, timezone

import nest_asyncio

# Patch Colab's pre-existing event loop so asyncio.run() works from cells
nest_asyncio.apply()


def run_async(coro):
    """Run an async coroutine synchronously inside a Colab cell."""
    return asyncio.get_event_loop().run_until_complete(coro)


# ── Test result registry ─────────────────────────────────────────────────────
_tests: dict = {}


def assert_test(name: str, condition: bool, detail: str = '') -> bool:
    """Record and print a single test assertion."""
    _tests[name] = {'passed': bool(condition), 'detail': detail}
    icon = '✅ PASS' if condition else '❌ FAIL'
    line = f'  {icon} │ {name}'
    if detail:
        line += f'\n           → {detail}'
    print(line)
    return bool(condition)


def section_banner(title: str) -> None:
    bar = '═' * 64
    print(f'\n{bar}')
    print(f'  {title}')
    print(f'{bar}\n')


print('✅ nest_asyncio applied.')
print('✅ Test tracker ready.')
print(f'   {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")}')


## Section 2 — Infrastructure Component Tests

Direct unit tests of the three core building blocks **before** touching the MCP server.


In [ ]:
# ─── Infrastructure component tests ──────────────────────────────────────────
section_banner('SECTION 2 — Infrastructure: Cache · CircuitBreaker · Fetcher')

from world_intel_mcp.cache import Cache
from world_intel_mcp.circuit_breaker import CircuitBreaker
from world_intel_mcp.fetcher import Fetcher

# ── Cache ─────────────────────────────────────────────────────────────────────
print('▸ Cache')
cache = Cache()
stats = cache.stats()
assert_test('Cache: stats() returns dict',       isinstance(stats, dict))
assert_test('Cache: has "total_entries" key',    'total_entries' in stats)
assert_test('Cache: has "db_path" key',          'db_path'       in stats)
print(f'  stats → {stats}\n')

# ── CircuitBreaker ────────────────────────────────────────────────────────────
print('▸ CircuitBreaker')
breaker = CircuitBreaker(failure_threshold=3, cooldown_seconds=300)
assert_test('CircuitBreaker: instantiated',               breaker is not None)
assert_test('CircuitBreaker: failure_threshold == 3',     breaker.failure_threshold == 3)
print()

# ── Fetcher ───────────────────────────────────────────────────────────────────
print('▸ Fetcher')
fetcher = Fetcher(cache=cache, breaker=breaker)
assert_test('Fetcher: has get_json',  hasattr(fetcher, 'get_json'))
assert_test('Fetcher: has get_text',  hasattr(fetcher, 'get_text'))
assert_test('Fetcher: has get_xml',   hasattr(fetcher, 'get_xml'))
assert_test('Fetcher: has close',     hasattr(fetcher, 'close'))
print()

sec2 = {k: v for k, v in _tests.items()
        if any(x in k for x in ('Cache', 'Circuit', 'Fetcher'))}
p = sum(1 for v in sec2.values() if v['passed'])
print(f'  Section 2: {p}/{len(sec2)} passed')


## Section 3 — MCP Server Protocol Test

Launches `world-intel-mcp` as a subprocess and connects via the **MCP stdio protocol**.  
Verifies server identity, enumerates all registered tools, and checks that news-relevant tools are present.


In [ ]:
# ─── MCP server protocol test ─────────────────────────────────────────────────
section_banner('SECTION 3 — MCP Server Protocol')

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Subprocess launch parameters — world-intel-mcp is on PATH after pip install
SERVER_PARAMS = StdioServerParameters(
    command='world-intel-mcp',
    args=[],
    env=dict(os.environ),
)

# The news/analysis tools we will exercise in subsequent sections
NEWS_TOOL_NAMES = {
    'intel_news_feed',
    'intel_trending_keywords',
    'intel_gdelt_search',
    'intel_extract_entities',
    'intel_classify_event',
    'intel_news_clusters',
    'intel_social_signals',
    'intel_hacker_news',
    'intel_status',
}


async def _test_server_protocol():
    print('Connecting to world-intel-mcp via stdio subprocess...')
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:

            # Initialise — exchange capabilities with the server
            init = await session.initialize()
            srv_name    = init.serverInfo.name
            srv_version = init.serverInfo.version

            assert_test('Server: name == "world-intel-mcp"',
                        srv_name == 'world-intel-mcp', f'Got: "{srv_name}"')
            assert_test('Server: version string present', bool(srv_version))
            print(f'  → {srv_name}  v{srv_version}\n')

            # Enumerate all registered tools
            tools_resp = await session.list_tools()
            tools      = tools_resp.tools
            tool_names = {t.name for t in tools}

            assert_test('Server: ≥ 87 tools registered',
                        len(tools) >= 87, f'got {len(tools)}')
            print(f'  → {len(tools)} tools registered\n')

            # Confirm every news-relevant tool is present
            print('  News & analysis tools:')
            for name in sorted(NEWS_TOOL_NAMES):
                present = name in tool_names
                assert_test(f'Tool present: {name}', present)
                if present:
                    t = next(x for x in tools if x.name == name)
                    desc = (t.description or '')[:60]
                    print(f'    ✅  {name:<35}  {desc}')

            return tools


all_tools = run_async(_test_server_protocol())

sec3 = {k: v for k, v in _tests.items() if 'Server:' in k or 'Tool present:' in k}
p = sum(1 for v in sec3.values() if v['passed'])
print(f'\n  Section 3: {p}/{len(sec3)} passed')


## Section 4 — `intel_news_feed`

Aggregates **80+ RSS feeds** across 24 categories with a 4-tier source ranking system.

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `category` | `str\|None` | `None` | Feed category key, or `None` for all categories |
| `limit` | `int` | `50` | Maximum items to return |

Valid categories: `geopolitics`, `security`, `technology`, `finance`, `military`, `science`, `think_tanks`, `middle_east`, `asia_pacific`, `africa`, `latin_america`, `energy`, `government`, `crisis`, `europe`, `health`, `maritime`, `space`, `nuclear`, `climate`, …


In [ ]:
# ─── Test intel_news_feed ─────────────────────────────────────────────────────
section_banner('SECTION 4 — intel_news_feed')


async def _test_news_feed():
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ── T1: no args — all categories, default limit ───────────────────
            print('▸ T1: no args (all categories, default limit=50)')
            r = await session.call_tool('intel_news_feed', {})
            d = json.loads(r.content[0].text)

            assert_test('news_feed: "items" is list',           isinstance(d.get('items'), list))
            assert_test('news_feed: count > 0',                 d.get('count', 0) > 0,
                        f'count={d.get("count")}')
            assert_test('news_feed: source == "rss-aggregator"',
                        d.get('source') == 'rss-aggregator')
            assert_test('news_feed: timestamp present',         bool(d.get('timestamp')))
            assert_test('news_feed: categories_fetched is list',
                        isinstance(d.get('categories_fetched'), list))

            if d.get('items'):
                item = d['items'][0]
                assert_test('news_feed item: has "title"',       'title'       in item)
                assert_test('news_feed item: has "link"',        'link'        in item)
                assert_test('news_feed item: has "published"',   'published'   in item)
                assert_test('news_feed item: has "feed_name"',   'feed_name'   in item)
                assert_test('news_feed item: has "source_tier"', 'source_tier' in item)

            cats_fetched = d.get('categories_fetched', [])
            print(f'  → {d.get("count")} items from {len(cats_fetched)} categories\n')

            # ── T2: geopolitics, limit=10 ─────────────────────────────────────
            print('▸ T2: category="geopolitics", limit=10')
            r2 = await session.call_tool('intel_news_feed',
                                         {'category': 'geopolitics', 'limit': 10})
            d2 = json.loads(r2.content[0].text)
            assert_test('news_feed[geo]: count ≤ 10',    d2.get('count', 0) <= 10,
                        f'count={d2.get("count")}')
            assert_test('news_feed[geo]: items is list', isinstance(d2.get('items'), list))
            print(f'  → {d2.get("count")} items\n')

            # ── T3: security, limit=5 ─────────────────────────────────────────
            print('▸ T3: category="security", limit=5')
            r3 = await session.call_tool('intel_news_feed',
                                         {'category': 'security', 'limit': 5})
            d3 = json.loads(r3.content[0].text)
            assert_test('news_feed[sec]: count ≤ 5', d3.get('count', 0) <= 5)
            print(f'  → {d3.get("count")} items\n')

            # ── T4: military, limit=5 ─────────────────────────────────────────
            print('▸ T4: category="military", limit=5')
            r4 = await session.call_tool('intel_news_feed',
                                         {'category': 'military', 'limit': 5})
            d4 = json.loads(r4.content[0].text)
            assert_test('news_feed[mil]: returned data', 'items' in d4 or 'error' in d4)
            print(f'  → {d4.get("count", 0)} items\n')

            # ── T5: crisis, limit=5 ───────────────────────────────────────────
            print('▸ T5: category="crisis", limit=5')
            r5 = await session.call_tool('intel_news_feed',
                                         {'category': 'crisis', 'limit': 5})
            d5 = json.loads(r5.content[0].text)
            assert_test('news_feed[crisis]: returned data', 'items' in d5 or 'error' in d5)
            print(f'  → {d5.get("count", 0)} items\n')

            # ── Display sample headlines ──────────────────────────────────────
            items_geo = d2.get('items', [])
            if items_geo:
                print('─' * 68)
                print('  Sample Headlines — geopolitics (limit=10)')
                print('─' * 68)
                for item in items_geo[:6]:
                    tier  = (item.get('source_tier') or 'unknown').rjust(11)
                    title = (item.get('title')       or '')[:66]
                    feed  = item.get('feed_name', '')
                    pub   = (item.get('published')   or '')[:10]
                    print(f'  [{tier}]  {title}')
                    print(f'               via {feed}  |  {pub}\n')

            return d, d2


_nf_all, _nf_geo = run_async(_test_news_feed())

sec4 = {k: v for k, v in _tests.items() if 'news_feed' in k}
p = sum(1 for v in sec4.values() if v['passed'])
print(f'  Section 4: {p}/{len(sec4)} passed')


## Section 5 — `intel_trending_keywords`

Fetches ~200 recent headlines, strips stopwords, and returns the **top-50 keywords by frequency**.  
Useful for detecting what topics are dominating the global news cycle right now.

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `min_count` | `int` | `3` | Minimum occurrences required to include a keyword |
| `hours` | `int` | `6` | Kept for API symmetry; RSS feeds are inherently recent |


In [ ]:
# ─── Test intel_trending_keywords ────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.facecolor'] = '#0A0E17'
matplotlib.rcParams['axes.facecolor']   = '#111827'
matplotlib.rcParams['text.color']       = '#E5E7EB'

section_banner('SECTION 5 — intel_trending_keywords')


async def _test_trending():
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ── T1: default args ──────────────────────────────────────────────
            print('▸ T1: default args (min_count=3)')
            r1 = await session.call_tool('intel_trending_keywords', {})
            d1 = json.loads(r1.content[0].text)

            assert_test('trending: "keywords" is list',
                        isinstance(d1.get('keywords'), list))
            assert_test('trending: total_items_analyzed > 0',
                        d1.get('total_items_analyzed', 0) > 0,
                        f'got {d1.get("total_items_analyzed")}')
            assert_test('trending: source == "keyword-analysis"',
                        d1.get('source') == 'keyword-analysis')
            assert_test('trending: timestamp present', bool(d1.get('timestamp')))

            kws = d1.get('keywords', [])
            if kws:
                assert_test('trending item: has "word"',
                            all('word'  in k for k in kws[:5]))
                assert_test('trending item: has "count"',
                            all('count' in k for k in kws[:5]))
                counts = [k['count'] for k in kws[:20]]
                assert_test('trending: sorted descending by count',
                             counts == sorted(counts, reverse=True))

            n_default = len(kws)
            print(f'  → {n_default} keywords from {d1.get("total_items_analyzed")} items\n')

            # ── T2: min_count=2 (expect more keywords) ────────────────────────
            print('▸ T2: min_count=2')
            r2 = await session.call_tool('intel_trending_keywords', {'min_count': 2})
            d2 = json.loads(r2.content[0].text)
            n_low = len(d2.get('keywords', []))
            assert_test('trending[min2]: has keywords', n_low > 0)
            print(f'  → {n_low} keywords\n')

            # ── T3: min_count=10 (expect fewer keywords) ──────────────────────
            print('▸ T3: min_count=10')
            r3 = await session.call_tool('intel_trending_keywords', {'min_count': 10})
            d3 = json.loads(r3.content[0].text)
            n_high = len(d3.get('keywords', []))
            assert_test('trending[min10]: fewer than min_count=2',
                        n_high <= n_low, f'{n_high} vs {n_low}')
            print(f'  → {n_high} keywords\n')

            # ── Bar chart: top-20 keywords ────────────────────────────────────
            chart_kws = kws[:20]
            if chart_kws:
                words = [k['word']  for k in chart_kws]
                cnts  = [k['count'] for k in chart_kws]

                fig, ax = plt.subplots(figsize=(12, 6))
                fig.patch.set_facecolor('#0A0E17')
                ax.set_facecolor('#111827')

                bars = ax.barh(words[::-1], cnts[::-1], color='#60A5FA', alpha=0.85)
                ax.set_xlabel('Occurrence count', color='#E5E7EB', labelpad=8)
                ax.set_title(
                    'Top 20 Trending Keywords — World-Intel-MCP Live News Feed',
                    color='#E5E7EB', fontsize=13, pad=12,
                )
                ax.tick_params(axis='both', colors='#E5E7EB')
                for spine in ax.spines.values():
                    spine.set_edgecolor('#374151')
                for bar, cnt in zip(bars, cnts[::-1]):
                    ax.text(bar.get_width() + 0.1,
                            bar.get_y() + bar.get_height() / 2,
                            str(cnt), va='center', color='#F59E0B', fontsize=9)

                plt.tight_layout()
                plt.savefig('trending_keywords.png', dpi=120,
                            bbox_inches='tight', facecolor='#0A0E17')
                plt.show()
                print('  📊 Chart saved: trending_keywords.png')

            return d1


_trending_data = run_async(_test_trending())

sec5 = {k: v for k, v in _tests.items() if 'trending' in k}
p = sum(1 for v in sec5.values() if v['passed'])
print(f'\n  Section 5: {p}/{len(sec5)} passed')


## Section 6 — `intel_gdelt_search`

Searches the **GDELT 2.0 Doc API** — no API key required.

| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `query` | `str` | `"conflict"` | Search terms passed to the GDELT API |
| `mode` | `str` | `"artlist"` | `"artlist"` → article list · `"timelinevol"` → volume over time |
| `limit` | `int` | `50` | Max articles returned in `artlist` mode |


In [ ]:
# ─── Test intel_gdelt_search ─────────────────────────────────────────────────
section_banner('SECTION 6 — intel_gdelt_search')


async def _test_gdelt():
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ── T1: artlist, query=conflict ───────────────────────────────────
            print('▸ T1: mode="artlist", query="conflict", limit=10')
            r1 = await session.call_tool('intel_gdelt_search',
                                         {'query': 'conflict', 'mode': 'artlist', 'limit': 10})
            d1 = json.loads(r1.content[0].text)

            if 'error' in d1:
                assert_test('gdelt[artlist]: returned (error noted)', True,
                             f'error={d1["error"][:80]}')
            else:
                assert_test('gdelt[artlist]: "articles" is list',
                             isinstance(d1.get('articles'), list))
                assert_test('gdelt[artlist]: count > 0',
                             d1.get('count', 0) > 0)
                assert_test('gdelt[artlist]: mode == "artlist"',
                             d1.get('mode') == 'artlist')
                assert_test('gdelt[artlist]: source == "gdelt"',
                             d1.get('source') == 'gdelt')
                if d1.get('articles'):
                    a = d1['articles'][0]
                    assert_test('gdelt article: has "title"',    'title'    in a)
                    assert_test('gdelt article: has "url"',      'url'      in a)
                    assert_test('gdelt article: has "domain"',   'domain'   in a)
                    assert_test('gdelt article: has "language"', 'language' in a)
                print(f'  → {d1.get("count")} articles\n')

            # ── T2: timelinevol, query=war ────────────────────────────────────
            print('▸ T2: mode="timelinevol", query="war"')
            r2 = await session.call_tool('intel_gdelt_search',
                                         {'query': 'war', 'mode': 'timelinevol'})
            d2 = json.loads(r2.content[0].text)

            if 'error' in d2:
                assert_test('gdelt[timeline]: returned (error noted)', True,
                             f'error={d2["error"][:80]}')
            else:
                assert_test('gdelt[timeline]: "timeline" is list',
                             isinstance(d2.get('timeline'), list))
                assert_test('gdelt[timeline]: mode == "timelinevol"',
                             d2.get('mode') == 'timelinevol')
                if d2.get('timeline'):
                    step = d2['timeline'][0]
                    assert_test('gdelt timeline step: has "date"', 'date' in step)
                    assert_test('gdelt timeline step: has "vol"',  'vol'  in step)
                print(f'  → {len(d2.get("timeline", []))} time steps\n')

            # ── T3: artlist, query=cyber attack ───────────────────────────────
            print('▸ T3: mode="artlist", query="cyber attack", limit=5')
            r3 = await session.call_tool('intel_gdelt_search',
                                         {'query': 'cyber attack',
                                          'mode': 'artlist', 'limit': 5})
            d3 = json.loads(r3.content[0].text)
            assert_test('gdelt[cyber]: returned data', 'articles' in d3 or 'error' in d3)
            print(f'  → {d3.get("count", 0)} articles\n')

            # ── T4: artlist, query=sanctions ──────────────────────────────────
            print('▸ T4: mode="artlist", query="sanctions", limit=5')
            r4 = await session.call_tool('intel_gdelt_search',
                                         {'query': 'sanctions',
                                          'mode': 'artlist', 'limit': 5})
            d4 = json.loads(r4.content[0].text)
            assert_test('gdelt[sanctions]: returned data', 'articles' in d4 or 'error' in d4)
            print(f'  → {d4.get("count", 0)} articles\n')

            # ── Display sample articles ───────────────────────────────────────
            if d1.get('articles'):
                print('─' * 68)
                print('  Sample GDELT Articles — "conflict"')
                print('─' * 68)
                for art in d1['articles'][:5]:
                    cc    = (art.get('sourcecountry') or '??')[:4]
                    title = (art.get('title')         or '')[:64]
                    dom   = art.get('domain', '')
                    date  = (art.get('seendate')      or '')[:10]
                    lang  = (art.get('language')      or '').upper()[:3]
                    print(f'  [{cc:>4}|{lang}]  {title}')
                    print(f'             {dom}  ·  {date}\n')

            # ── Timeline chart ────────────────────────────────────────────────
            timeline = d2.get('timeline', [])
            if len(timeline) > 2:
                dates = [t['date'] for t in timeline]
                vols  = [t.get('vol', 0) for t in timeline]

                fig, ax = plt.subplots(figsize=(14, 4))
                fig.patch.set_facecolor('#0A0E17')
                ax.set_facecolor('#111827')
                ax.fill_between(range(len(vols)), vols, alpha=0.35, color='#60A5FA')
                ax.plot(range(len(vols)), vols, color='#60A5FA', linewidth=1.8)

                step = max(1, len(dates) // 8)
                ax.set_xticks(range(0, len(dates), step))
                ax.set_xticklabels(
                    [dates[i][:10] for i in range(0, len(dates), step)],
                    rotation=35, ha='right', color='#E5E7EB', fontsize=8,
                )
                ax.set_ylabel('Coverage volume', color='#E5E7EB')
                ax.set_title('GDELT Timeline Volume — "war"',
                             color='#E5E7EB', fontsize=13, pad=12)
                ax.tick_params(colors='#E5E7EB')
                for spine in ax.spines.values():
                    spine.set_edgecolor('#374151')

                plt.tight_layout()
                plt.savefig('gdelt_timeline.png', dpi=120,
                            bbox_inches='tight', facecolor='#0A0E17')
                plt.show()
                print('  📊 Chart saved: gdelt_timeline.png')

            return d1, d2


_gdelt_artlist, _gdelt_timeline = run_async(_test_gdelt())

sec6 = {k: v for k, v in _tests.items() if 'gdelt' in k}
p = sum(1 for v in sec6.values() if v['passed'])
print(f'\n  Section 6: {p}/{len(sec6)} passed')


## Section 7 — NLP & Analysis Tools

Tests the analysis tools that transform raw text into structured intelligence.

| Tool | Description |
|------|-------------|
| `intel_extract_entities` | Named entity extraction: countries, leaders, orgs, CVEs, APTs |
| `intel_classify_event` | 14-category threat classification + severity score 1–10 |
| `intel_news_clusters` | Jaccard similarity topic clustering across recent headlines |
| `intel_social_signals` | Reddit geopolitical discussion velocity |
| `intel_hacker_news` | Hacker News top stories (Firebase API) |
| `intel_status` | Server health, SQLite cache stats, circuit breaker states |


In [ ]:
# ─── Test NLP & analysis tools ───────────────────────────────────────────────
section_banner('SECTION 7 — NLP & Analysis Tools')

SAMPLE_TEXT = (
    'Russian drone strikes hit Ukrainian energy grid in Kyiv; '
    'NATO Secretary General Stoltenberg convenes emergency summit. '
    'FSB claims coordinated cyber operation targeting EU financial systems via SWIFT.'
)
CONFLICT_TEXT = (
    'Iranian-backed militia launched ballistic missile attack on US forces '
    'at Al-Asad air base in Iraq, wounding 12 soldiers; Pentagon confirms incident.'
)


async def _test_nlp():
    results = {}
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ── intel_extract_entities ────────────────────────────────────────
            print('▸ intel_extract_entities')
            r1 = await session.call_tool('intel_extract_entities', {'text': SAMPLE_TEXT})
            d1 = json.loads(r1.content[0].text)
            results['entities'] = d1
            assert_test('extract_entities: returns dict',         isinstance(d1, dict))
            assert_test('extract_entities: non-empty',            len(d1) > 0)
            print(f'  keys → {list(d1.keys())}')
            for k, v in d1.items():
                if isinstance(v, list) and v:
                    print(f'  {k}: {v[:4]}')
            print()

            # ── intel_classify_event ──────────────────────────────────────────
            print('▸ intel_classify_event')
            r2 = await session.call_tool('intel_classify_event', {'text': CONFLICT_TEXT})
            d2 = json.loads(r2.content[0].text)
            results['classify'] = d2
            assert_test('classify_event: returns dict',           isinstance(d2, dict))
            assert_test('classify_event: non-empty',              len(d2) > 0)
            if 'severity' in d2:
                sev = d2['severity']
                assert_test('classify_event: severity in 1–10',
                             isinstance(sev, (int, float)) and 1 <= float(sev) <= 10,
                             f'got {sev}')
            print(f'  → {d2}\n')

            # ── intel_news_clusters ───────────────────────────────────────────
            print('▸ intel_news_clusters')
            r3 = await session.call_tool('intel_news_clusters', {})
            d3 = json.loads(r3.content[0].text)
            results['clusters'] = d3
            assert_test('news_clusters: returns data',            isinstance(d3, (dict, list)))
            if isinstance(d3, dict) and 'clusters' in d3:
                assert_test('news_clusters: "clusters" is list',  isinstance(d3['clusters'], list))
                print(f'  → {len(d3["clusters"])} clusters\n')
            else:
                assert_test('news_clusters: non-empty response',  bool(d3))
                print(f'  → type={type(d3).__name__}\n')

            # ── intel_social_signals ──────────────────────────────────────────
            print('▸ intel_social_signals')
            r4 = await session.call_tool('intel_social_signals', {})
            d4 = json.loads(r4.content[0].text)
            results['social'] = d4
            assert_test('social_signals: returns data',           isinstance(d4, (dict, list)))
            keys = list(d4.keys())[:6] if isinstance(d4, dict) else []
            print(f'  → keys={keys}\n')

            # ── intel_hacker_news ─────────────────────────────────────────────
            print('▸ intel_hacker_news')
            r5 = await session.call_tool('intel_hacker_news', {})
            d5 = json.loads(r5.content[0].text)
            results['hackernews'] = d5
            assert_test('hacker_news: returns data',              isinstance(d5, (dict, list)))
            if isinstance(d5, dict) and 'stories' in d5:
                assert_test('hacker_news: "stories" is list',     isinstance(d5['stories'], list))
                print(f'  → {len(d5["stories"])} stories')
                for s in (d5['stories'] or [])[:3]:
                    title = (s.get('title') if isinstance(s, dict) else str(s))[:68]
                    print(f'    • {title}')
            else:
                assert_test('hacker_news: non-empty',             bool(d5))
                print(f'  → type={type(d5).__name__}')
            print()

            # ── intel_status ──────────────────────────────────────────────────
            print('▸ intel_status')
            r6 = await session.call_tool('intel_status', {})
            d6 = json.loads(r6.content[0].text)
            results['status'] = d6
            assert_test('server_status: returns dict',            isinstance(d6, dict))
            print(f'  → keys={list(d6.keys())[:8]}\n')

            # ── Entity extraction summary ─────────────────────────────────────
            print('─' * 68)
            print('  Entity Extraction Summary')
            print('─' * 68)
            print(f'  Input text:\n  "{SAMPLE_TEXT[:90]}..."\n')
            for k, v in d1.items():
                if isinstance(v, list) and v:
                    print(f'  {k:20}: {v[:5]}')
                elif isinstance(v, (str, int, float)) and v:
                    print(f'  {k:20}: {v}')

            return results


_nlp_results = run_async(_test_nlp())

sec7 = {k: v for k, v in _tests.items()
        if any(x in k for x in ('extract', 'classify', 'cluster', 'social', 'hacker', 'server_status'))}
p = sum(1 for v in sec7.values() if v['passed'])
print(f'\n  Section 7: {p}/{len(sec7)} passed')


## Section 8 — WorldIntelAgent

An **Anthropic tool-use agent** that routes queries through the World-Intel-MCP server.

### How it works
1. Opens an MCP `stdio_client` session to `world-intel-mcp`
2. Converts MCP `Tool` objects → Anthropic tool definitions (camelCase → snake_case schema key)
3. Sends the user query to Claude with those tools available
4. On `tool_use` blocks: dispatches the call back through the live MCP session
5. Loops (up to `max_iterations`) until `stop_reason == "end_turn"`

### Requires
- `ANTHROPIC_API_KEY` in Colab Secrets (🔑) **or** `os.environ['ANTHROPIC_API_KEY']`


In [ ]:
# ─── WorldIntelAgent class ────────────────────────────────────────────────────
import anthropic
from typing import Any

section_banner('SECTION 8 — WorldIntelAgent Class')

# Tools exposed to the agent in the demo (subset focused on news intelligence)
AGENT_TOOLS = {
    'intel_news_feed',
    'intel_trending_keywords',
    'intel_gdelt_search',
    'intel_extract_entities',
    'intel_classify_event',
    'intel_news_clusters',
    'intel_social_signals',
    'intel_hacker_news',
}


def _mcp_to_anthropic_tool(tool: Any) -> dict:
    """Convert an MCP Tool object → Anthropic tool definition dict."""
    raw = tool.inputSchema if hasattr(tool, 'inputSchema') else {}
    # Materialise Pydantic models if present
    if hasattr(raw, 'model_dump'):
        schema = raw.model_dump()
    elif hasattr(raw, 'dict'):
        schema = raw.dict()
    else:
        schema = dict(raw) if raw else {}
    if not schema or 'type' not in schema:
        schema = {'type': 'object', 'properties': {}, 'required': []}
    return {
        'name':         tool.name,
        'description':  tool.description or '',
        'input_schema': schema,
    }


class WorldIntelAgent:
    """
    Anthropic tool-use agent backed by the World-Intel-MCP MCP server.

    Example:
        agent = WorldIntelAgent(api_key='sk-ant-...')
        result = run_async(agent.run('Top security headlines?'))
        print(result['answer'])
    """

    def __init__(
        self,
        api_key: str,
        model: str = 'claude-sonnet-4-6',
        allowed_tools: set | None = None,
        max_iterations: int = 10,
        verbose: bool = True,
    ) -> None:
        self.client        = anthropic.Anthropic(api_key=api_key)
        self.model         = model
        self.allowed_tools = allowed_tools or AGENT_TOOLS
        self.max_iters     = max_iterations
        self.verbose       = verbose

    def _log(self, msg: str) -> None:
        if self.verbose:
            print(msg)

    async def run(self, user_query: str) -> dict:
        """
        Execute one agentic turn.

        Returns dict with keys:
          answer      — Claude's final synthesised text response
          tool_calls  — list of {name, input, result_preview}
          iterations  — number of Claude API calls made
        """
        tool_log: list[dict] = []

        async with stdio_client(SERVER_PARAMS) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                # Fetch tool list from the live MCP server
                tools_resp     = await session.list_tools()
                claude_tools   = [
                    _mcp_to_anthropic_tool(t)
                    for t in tools_resp.tools
                    if t.name in self.allowed_tools
                ]

                self._log(f'\n{"═"*64}')
                self._log(f'  QUERY : {user_query}')
                self._log(f'  MODEL : {self.model}')
                self._log(f'  TOOLS : {len(claude_tools)} available')
                self._log(f'{"═"*64}')

                messages = [{'role': 'user', 'content': user_query}]

                for iteration in range(1, self.max_iters + 1):
                    response = self.client.messages.create(
                        model      = self.model,
                        max_tokens = 4096,
                        system     = (
                            'You are a world intelligence analyst. '
                            'You have access to real-time news feeds, RSS aggregators, '
                            'the GDELT global news database, and NLP analysis tools. '
                            'Always use the available tools to answer with current, '
                            'evidence-based information. Cite your sources.'
                        ),
                        tools    = claude_tools,
                        messages = messages,
                    )

                    self._log(f'\n  [Iter {iteration}] stop_reason={response.stop_reason}')

                    for block in response.content:
                        if block.type == 'text' and block.text:
                            snippet = block.text[:240]
                            self._log(f'\n  📝 {snippet}{"..." if len(block.text)>240 else ""}')

                    # ── Final answer ──────────────────────────────────────────
                    if response.stop_reason == 'end_turn':
                        answer = ' '.join(
                            b.text for b in response.content
                            if b.type == 'text' and b.text
                        )
                        return {'answer': answer, 'tool_calls': tool_log,
                                'iterations': iteration}

                    # ── Tool dispatch ─────────────────────────────────────────
                    if response.stop_reason == 'tool_use':
                        tool_results = []
                        for block in response.content:
                            if block.type != 'tool_use':
                                continue
                            self._log(f'\n  🔧 {block.name}  in={json.dumps(block.input)[:120]}')
                            mcp_r   = await session.call_tool(block.name, block.input)
                            txt     = mcp_r.content[0].text if mcp_r.content else '{}'
                            preview = txt[:280] + ('...' if len(txt) > 280 else '')
                            self._log(f'     out={preview}')
                            tool_log.append({'name':           block.name,
                                             'input':          block.input,
                                             'result_preview': preview})
                            tool_results.append({'type':        'tool_result',
                                                 'tool_use_id': block.id,
                                                 'content':     txt})
                        messages.append({'role': 'assistant', 'content': response.content})
                        messages.append({'role': 'user',      'content': tool_results})
                    else:
                        self._log(f'  ⚠ Unexpected stop_reason: {response.stop_reason}')
                        break

        return {'answer': 'Max iterations reached without final answer.',
                'tool_calls': tool_log, 'iterations': self.max_iters}


print('✅ WorldIntelAgent defined.')
print(f'   Allowed tools ({len(AGENT_TOOLS)}):')
for t in sorted(AGENT_TOOLS):
    print(f'   • {t}')


In [ ]:
# ─── API key setup ────────────────────────────────────────────────────────────
# Option A (recommended): add ANTHROPIC_API_KEY to Colab Secrets
#   Click the 🔑 key icon → add secret → toggle notebook access on
#
# Option B (manual override):
# import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'  # uncomment + fill in

ANTHROPIC_API_KEY = ''

# Try Colab Secrets first
try:
    from google.colab import userdata  # type: ignore[import]
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY') or ''
    if ANTHROPIC_API_KEY:
        print('✅ API key loaded from Colab Secrets.')
except Exception:
    pass

# Fall back to environment variable
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
    if ANTHROPIC_API_KEY:
        print('✅ API key loaded from environment variable.')

AGENT_ENABLED = bool(ANTHROPIC_API_KEY)

if AGENT_ENABLED:
    masked = ANTHROPIC_API_KEY[:8] + '...' + ANTHROPIC_API_KEY[-4:]
    print(f'  Key: {masked}')
    print('\n✅ Agent demo is ENABLED — proceed to Section 9.')
else:
    print('⚠️  ANTHROPIC_API_KEY not found.')
    print('   The agent demo (Section 9) will be skipped.')
    print('   To enable:')
    print('   • Add ANTHROPIC_API_KEY to Colab Secrets (🔑 icon), OR')
    print('   • os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."')


## Section 9 — Agent Demo: End-to-End Intelligence Queries

Runs **three progressively complex queries** through `WorldIntelAgent`, demonstrating real agentic tool chaining against live news data.

| Query | Likely tools used |
|-------|-------------------|
| Top geopolitics headlines | `intel_news_feed` |
| Trending security topics + driver analysis | `intel_trending_keywords` + `intel_news_feed` |
| Cyber attack search → classify → extract entities | `intel_gdelt_search` + `intel_classify_event` + `intel_extract_entities` |

> **Requires `ANTHROPIC_API_KEY`** — set in Section 8 before running.


In [ ]:
# ─── Agent demo ───────────────────────────────────────────────────────────────
section_banner('SECTION 9 — Agent Demo')

if not AGENT_ENABLED:
    print('⚠️  Skipping — ANTHROPIC_API_KEY not set.')
    print('   Set the key in Section 8 and re-run this cell.')
else:
    agent = WorldIntelAgent(api_key=ANTHROPIC_API_KEY, verbose=True)
    _agent_results = []

    # ─────────────────────────────────────────────────────────────────────────
    # Query 1 — Top geopolitics headlines
    # ─────────────────────────────────────────────────────────────────────────
    print('\n' + '─'*64)
    print('  QUERY 1: Top geopolitics headlines')
    print('─'*64)
    r1 = run_async(agent.run(
        'What are the top geopolitics headlines right now? '
        'Give me a concise intelligence summary with source attribution.'
    ))
    _agent_results.append(r1)

    assert_test('agent[q1]: answer is non-empty string',
                isinstance(r1.get('answer'), str) and len(r1.get('answer', '')) > 20)
    assert_test('agent[q1]: at least 1 tool called',
                len(r1.get('tool_calls', [])) >= 1)

    print(f'\n  Tools used : {[tc["name"] for tc in r1.get("tool_calls", [])]}')
    print(f'  Iterations : {r1.get("iterations")}')
    print(f'\n  ┌─ ANSWER {"─"*55}')
    for line in r1.get('answer', '')[:1500].split('\n'):
        print(f'  │  {line}')
    print(f'  └{"─"*64}')

    # ─────────────────────────────────────────────────────────────────────────
    # Query 2 — Trending security topics
    # ─────────────────────────────────────────────────────────────────────────
    print('\n\n' + '─'*64)
    print('  QUERY 2: Trending security topics + driver analysis')
    print('─'*64)
    r2 = run_async(agent.run(
        'What topics are currently trending in global security and geopolitics news? '
        'Identify the top spiking keywords and explain what underlying stories are driving them.'
    ))
    _agent_results.append(r2)

    assert_test('agent[q2]: answer is non-empty string',
                isinstance(r2.get('answer'), str) and len(r2.get('answer', '')) > 20)
    assert_test('agent[q2]: at least 1 tool called',
                len(r2.get('tool_calls', [])) >= 1)

    print(f'\n  Tools used : {[tc["name"] for tc in r2.get("tool_calls", [])]}')
    print(f'  Iterations : {r2.get("iterations")}')
    print(f'\n  ┌─ ANSWER {"─"*55}')
    for line in r2.get('answer', '')[:1500].split('\n'):
        print(f'  │  {line}')
    print(f'  └{"─"*64}')

    # ─────────────────────────────────────────────────────────────────────────
    # Query 3 — Cyber attack intelligence chain
    # ─────────────────────────────────────────────────────────────────────────
    print('\n\n' + '─'*64)
    print('  QUERY 3: Cyber attack search → classify → entity extraction')
    print('─'*64)
    r3 = run_async(agent.run(
        'Search GDELT for the latest news about cyber attacks. '
        'Take the most relevant article, classify it by threat type and severity (1-10), '
        'then extract the key actors, countries, and organisations involved.'
    ))
    _agent_results.append(r3)

    assert_test('agent[q3]: answer is non-empty string',
                isinstance(r3.get('answer'), str) and len(r3.get('answer', '')) > 20)
    assert_test('agent[q3]: at least 1 tool called',
                len(r3.get('tool_calls', [])) >= 1)

    print(f'\n  Tools used : {[tc["name"] for tc in r3.get("tool_calls", [])]}')
    print(f'  Iterations : {r3.get("iterations")}')
    print(f'\n  ┌─ ANSWER {"─"*55}')
    for line in r3.get('answer', '')[:1500].split('\n'):
        print(f'  │  {line}')
    print(f'  └{"─"*64}')

    sec9 = {k: v for k, v in _tests.items() if 'agent[' in k}
    p = sum(1 for v in sec9.values() if v['passed'])
    print(f'\n\n  Section 9: {p}/{len(sec9)} passed')


## Final Test Summary

Pass/fail tally across every assertion in the notebook.


In [ ]:
# ─── Final test summary ───────────────────────────────────────────────────────
section_banner('FINAL TEST SUMMARY — World-Intel-MCP')

total  = len(_tests)
passed = sum(1 for v in _tests.values() if v['passed'])
failed = total - passed

# ── Group by section prefix ───────────────────────────────────────────────────
GROUPS = [
    ('Cache:',          'Sec 2 — Cache'),
    ('Circuit',         'Sec 2 — CircuitBreaker'),
    ('Fetcher:',        'Sec 2 — Fetcher'),
    ('Server:',         'Sec 3 — MCP Server Identity'),
    ('Tool present:',   'Sec 3 — MCP Tool Registry'),
    ('news_feed',       'Sec 4 — intel_news_feed'),
    ('trending',        'Sec 5 — intel_trending_keywords'),
    ('gdelt',           'Sec 6 — intel_gdelt_search'),
    ('extract_entities','Sec 7 — intel_extract_entities'),
    ('classify_event',  'Sec 7 — intel_classify_event'),
    ('news_clusters',   'Sec 7 — intel_news_clusters'),
    ('social_signals',  'Sec 7 — intel_social_signals'),
    ('hacker_news',     'Sec 7 — intel_hacker_news'),
    ('server_status',   'Sec 7 — intel_status'),
    ('agent[',          'Sec 9 — Agent Demo'),
]

for prefix, label in GROUPS:
    grp = {k: v for k, v in _tests.items() if k.startswith(prefix) or prefix in k}
    if not grp:
        continue
    gp = sum(1 for v in grp.values() if v['passed'])
    gt = len(grp)
    icon = '✅' if gp == gt else ('⚠️ ' if gp > 0 else '❌')
    print(f'  {icon}  {label:<42}  {gp}/{gt}')

pct = round(passed / total * 100) if total else 0
print(f'\n  {"─"*64}')
print(f'  TOTAL  {passed}/{total} passed ({pct}%)   {failed} failed')
print(f'  {"─"*64}')

if failed:
    print('\n  ❌ Failed tests:')
    for name, r in _tests.items():
        if not r['passed']:
            detail = f' → {r["detail"]}' if r.get('detail') else ''
            print(f'    • {name}{detail}')

print(f'\n  Completed : {datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")}')
print(f'  Source    : https://github.com/marc-shade/world-intel-mcp')
